# FindMate QnA Bot using MiniLM + FAISS
This notebook follows the same pipeline as HadithBot, but the topic is FindMate Campus Lost & Found.

## Step 1: Import libraries

In [19]:
import pandas as pd
import numpy as np
import re

## Step 2: Load QnA dataset

In [20]:
qna_df = pd.read_csv("qna_dataset.csv")
qna_df.head()

,question,answer,category
0,Hello,👋 Hello! Welcome to FindMate. Tell me what you...,greeting
1,Hi FindMate,"👋 Hi! I can help with lost items, found items,...",greeting
2,I lost my ID card,📋 Looking for your ID card? Check Student Affa...,lost_id
3,My student card is missing,"📋 For a missing student card, visit Student Af...",lost_id
4,Where should I go for lost ID card,"📋 Please check Student Affairs Office, Library...",lost_id


## Step 3: Preprocess the text

In [21]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

qna_df['clean_question'] = qna_df['question'].apply(clean_text)
qna_df['clean_answer'] = qna_df['answer'].apply(clean_text)
qna_df['clean_text'] = qna_df['clean_question'] + ' ' + qna_df['clean_answer']
qna_df.head()

,question,answer,category,clean_question,clean_answer,clean_text
0,Hello,👋 Hello! Welcome to FindMate. Tell me what you...,greeting,hello,hello welcome to findmate tell me what you los...,hello hello welcome to findmate tell me what y...
1,Hi FindMate,"👋 Hi! I can help with lost items, found items,...",greeting,hi findmate,hi i can help with lost items found items cont...,hi findmate hi i can help with lost items foun...
2,I lost my ID card,📋 Looking for your ID card? Check Student Affa...,lost_id,i lost my id card,looking for your id card check student affairs...,i lost my id card looking for your id card che...
3,My student card is missing,"📋 For a missing student card, visit Student Af...",lost_id,my student card is missing,for a missing student card visit student affai...,my student card is missing for a missing stude...
4,Where should I go for lost ID card,"📋 Please check Student Affairs Office, Library...",lost_id,where should i go for lost id card,please check student affairs office library re...,where should i go for lost id card please chec...


## Step 4: Save cleaned data

In [22]:
qna_df.to_csv('cleaned_findmate_qna.csv', index=False)

## Step 5: Install Sentence Transformers

In [23]:
%pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Step 6: Load Hugging Face MiniLM model

In [24]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

## Step 7: Generate embeddings

In [25]:
embeddings = model.encode(qna_df['clean_text'].values)
embeddings = np.array(embeddings).astype('float32')
embeddings.shape

(35, 384)

## Step 8: Save embeddings

In [26]:
np.save('findmate_embeddings.npy', embeddings)

## Step 9: Load embeddings again

In [27]:
embeddings = np.load('findmate_embeddings.npy').astype('float32')

## Step 10: Install FAISS

In [28]:
%pip install faiss-cpu

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Step 11: Create FAISS index

In [29]:
import faiss

dimensions = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimensions)

## Step 12: Add embeddings to FAISS

In [30]:
faiss_index.add(embeddings)

## Step 13: Save FAISS index

In [31]:
faiss.write_index(faiss_index, 'findmate_faiss.index')

## Step 14: Similarity search function

In [32]:
def get_similar_findmate_answer(query, count=3, model=model, faiss_index=faiss_index):
    query_clean = clean_text(query)
    query_embedding = model.encode([query_clean])
    query_embedding = np.array(query_embedding).astype('float32')

    distance, indices = faiss_index.search(query_embedding, count)

    for i in range(count):
        row_no = indices[0][i]
        print(f"Result {i+1}, Distance: {distance[0][i]}")
        print("Question:", qna_df['question'].iloc[row_no])
        print("Answer:", qna_df['answer'].iloc[row_no])
        print("Category:", qna_df['category'].iloc[row_no])
        print("-" * 50)

## Step 15: Test the bot

In [33]:
get_similar_findmate_answer("I lost my student card")

Result 1, Distance: 0.5218391418457031
Question: My student card is missing
Answer: 📋 For a missing student card, visit Student Affairs Office first. Also check Library Reception and Security Office.
Category: lost_id
--------------------------------------------------
Result 2, Distance: 0.5539546012878418
Question: Where should I go for lost ID card
Answer: 📋 Please check Student Affairs Office, Library Reception and Security Office for your lost ID card.
Category: lost_id
--------------------------------------------------
Result 3, Distance: 0.5778257250785828
Question: I found a student card
Answer: ✅ Submit the student card to Student Affairs Office or Library Reception so it can be returned to the owner.
Category: found_id
--------------------------------------------------


In [34]:
get_similar_findmate_answer("where should I submit a found wallet")

Result 1, Distance: 0.9693002700805664
Question: I found a wallet
Answer: ✅ Wonderful! Please submit the wallet to Security Office or Student Affairs Office. Do not open it or share personal details.
Category: found_wallet
--------------------------------------------------
Result 2, Distance: 0.9950006008148193
Question: My wallet is missing
Answer: 💳 If your wallet is missing, report it to Security Office quickly and also check Student Affairs Office.
Category: lost_wallet
--------------------------------------------------
Result 3, Distance: 1.137700080871582
Question: I lost my wallet
Answer: 💳 Lost wallet? Check Security Office first, then Student Affairs Office, Cafeteria/Canteen and your last classroom.
Category: lost_wallet
--------------------------------------------------
